# Air quality log postprocessing

Merges a run's hourly logger CSVs into one continuous time series and plots the
variables the SEN65 produces.

Each file written by [`main.py`](main.py) / [`log_aq.py`](log_aq.py) stands alone: a
`# Start:` header holding the only absolute time in the file, and one row per second
carrying `elapsed_s` — whole seconds since that start. Absolute time is
`Start + elapsed_s`, and because the filename, the header and the offset origin all
come from one reading of the clock, hourly files stitch back together without a seam.

What this notebook does:

1. Reads every CSV in a directory, recovering each file's start time, clock
   provenance and location from its header.
2. Merges them onto **one continuous time axis** spanning the whole run,
   measuring the logger's actual sampling cadence and marking real outages as
   explicit breaks — so plots break where the logger was off instead of drawing
   a line across the hole.
3. Reports coverage, gaps and any file whose clock the device flagged as
   untrustworthy.
4. Plots PM, the VOC/NOx indices, a per-variable overview, the hour-of-day
   profile and the distributions.
5. Writes the merged series back out as a single CSV.

## 1. Configuration

Point `LOG_DIR` at the folder holding the downloaded CSVs. Everything else has a
working default.

In [ ]:
from pathlib import Path

# Where the hourly CSVs live (the folder you unzipped the "download all" into).
LOG_DIR = Path("/home/meso/dev/aq-logs/20260903")
PATTERN = "*.csv"

# The logger writes one row per sensor sample, and the SEN65 does not hand samples
# over on a perfectly regular beat. "auto" measures the typical interval from the data
# and keeps every sample at the time it was taken; a pandas frequency ("1s", "2s")
# forces the samples onto a regular grid instead, dropping any that miss it.
SAMPLE_PERIOD = "auto"

# A jump longer than this many typical intervals counts as the logger being off, and
# becomes a break in the series. Anything shorter is the sensor's own jitter.
GAP_FACTOR = 3

# Plots resample to this for legibility — set to None to plot every sample.
RESAMPLE = "1min"

# The device writes local wall-clock times with no timezone, so the merged index is
# tz-naive. Set e.g. "Australia/Melbourne" to localize it (does not shift the times,
# only labels them).
TIMEZONE = None

# Drop files whose "# Clock:" header says the RTC was unset or absent. Their start
# times — and therefore their position on the merged axis — cannot be believed.
REQUIRE_TRUSTED_CLOCK = False

# WHO 2021 24-hour air quality guideline levels, drawn as reference lines.
SHOW_GUIDELINES = True
WHO_24H = {"PM2.5_ug_m3": 15.0, "PM10_ug_m3": 45.0}

# Matplotlib styling: set True if your notebook theme is dark.
DARK = False

# Save every figure to FIG_DIR as well as showing it.
SAVE_FIGS = False
FIG_DIR = Path("figures")

MERGED_CSV = Path("merged.csv")

## 2. Reading a single log file

The header carries everything the rows do not: where the logger was, when it
started, and whether the clock that said so can be trusted. A file with no
`# Start:` line — hand-edited, or truncated — falls back to its filename, which
encodes the same instant.

In [ ]:
import re
import warnings

import numpy as np
import pandas as pd

def plural(n, word):
    return f"{n} {word}" if n == 1 else f"{n} {word}s"


_HEADER_RE = re.compile(r"^#\s*([A-Za-z]+)\s*:\s*(.*)$")
_TIME_RE = re.compile(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}")
_NAME_RE = re.compile(r"(\d{8})_(\d{6})(?:-(\d+))?$")


def read_header(path):
    """Return the '# key: value' lines at the top of a log as a dict."""
    meta = {}
    with open(path, "r", errors="replace") as fh:
        for line in fh:
            if not line.startswith("#"):
                break
            m = _HEADER_RE.match(line.strip())
            if m:
                meta[m.group(1).lower()] = m.group(2).strip()
    return meta


def start_time(path, meta):
    """The file's absolute t0: the '# Start:' header, else its filename."""
    m = _TIME_RE.search(meta.get("start", ""))
    if m:
        return pd.Timestamp(m.group(0))
    m = _NAME_RE.search(Path(path).stem)
    if m:
        return pd.Timestamp(f"{m.group(1)} {m.group(2)}")
    raise ValueError(f"{path}: no '# Start:' header and no timestamp in the filename")


def location(meta):
    """(lat, lon) from a '# Location:' header, or (None, None)."""
    text = meta.get("location", "")
    lat = re.search(r"lat\s*=\s*(-?[\d.]+)", text)
    lon = re.search(r"lon\s*=\s*(-?[\d.]+)", text)
    return (float(lat.group(1)) if lat else None,
            float(lon.group(1)) if lon else None)


def clock_trusted(meta):
    """False when the device itself flagged the start time as unreliable."""
    return "unreliable" not in meta.get("clock", "").lower()


def read_log(path):
    """Read one logger CSV. Returns (DataFrame indexed by timestamp, metadata dict)."""
    meta = read_header(path)
    t0 = start_time(path, meta)
    lat, lon = location(meta)

    df = pd.read_csv(path, comment="#")
    if "elapsed_s" not in df.columns:
        raise ValueError(f"{path}: no elapsed_s column — is this a logger CSV?")

    # Every column is numeric; blanks are readings the sensor did not supply
    # (NOx in particular stays empty while its index warms up).
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["elapsed_s"])

    df.index = t0 + pd.to_timedelta(df.pop("elapsed_s"), unit="s")
    df.index.name = "timestamp"

    info = {
        "file": Path(path).name,
        "start": t0,
        "end": df.index[-1] if len(df) else t0,
        "rows": len(df),
        "clock": meta.get("clock", "(not recorded)"),
        "clock_ok": clock_trusted(meta),
        "lat": lat,
        "lon": lon,
    }
    return df, info

## 3. Loading the run

Files are read in start-time order, not filename order — they usually agree, but a
log whose name was deduplicated (`…-1.csv`, written when the clock was stopped)
does not sort where its data belongs.

In [ ]:
paths = sorted(Path(LOG_DIR).glob(PATTERN))
if not paths:
    raise FileNotFoundError(
        f"No files matching {PATTERN!r} in {Path(LOG_DIR).resolve()}.\n"
        "Point LOG_DIR at the folder you unzipped the logger download into."
    )

frames, infos, skipped = [], [], []
for path in paths:
    try:
        df, info = read_log(path)
    except Exception as exc:                      # a bad file should not sink the run
        skipped.append((path.name, str(exc)))
        continue
    if info["rows"] == 0:
        skipped.append((path.name, "no data rows"))
        continue
    if REQUIRE_TRUSTED_CLOCK and not info["clock_ok"]:
        skipped.append((path.name, f"untrusted clock: {info['clock']}"))
        continue
    frames.append(df)
    infos.append(info)

if not frames:
    raise ValueError("Every file was skipped — see `skipped` for why.")

order = np.argsort([i["start"] for i in infos])
frames = [frames[i] for i in order]
infos = [infos[i] for i in order]

inventory = pd.DataFrame(infos).set_index("file")
inventory["duration"] = inventory["end"] - inventory["start"]

for name, why in skipped:
    print(f"skipped {name}: {why}")
if not inventory["clock_ok"].all():
    bad = inventory.index[~inventory["clock_ok"]].tolist()
    print(f"\nWARNING: {len(bad)} file(s) have an untrusted clock — their start "
          f"times, and so their place on the merged axis, may be wrong:")
    for name in bad:
        print(f"  {name}: {inventory.loc[name, 'clock']}")

inventory[["start", "end", "duration", "rows", "clock_ok"]]

## 4. Merging onto one time axis

Concatenating the files is the easy half. The half that matters is the index.

The logger writes a row per sensor sample, not per second: the SEN65 hands over a
measurement when it has one, so consecutive rows sit a couple of seconds apart and
that spacing wanders. The merge therefore **measures the cadence** instead of
assuming it, and treats only a jump much longer than the typical interval —
`GAP_FACTOR` times it — as the logger being off. Each of those outages gets one
explicit empty row, so every plot below breaks over it rather than drawing a
straight line across an hour that was never recorded, while ordinary jitter is left
alone.

Forcing a regular grid is still one config line away: set `SAMPLE_PERIOD` to a
frequency like `"1s"` and the samples are reindexed onto it, with a warning naming
any that did not land on a slot.

Duplicate timestamps — a file re-downloaded, two runs of the same second, or the
clock stepping back under a resync — keep the last occurrence.

In [ ]:
raw = pd.concat(frames).sort_index()

duplicates = int(raw.index.duplicated().sum())
if duplicates:
    print(f"{plural(duplicates, 'duplicate timestamp')} collapsed, keeping the last of each")
raw = raw[~raw.index.duplicated(keep="last")]

if TIMEZONE:
    raw.index = raw.index.tz_localize(TIMEZONE)

# What the cadence actually is, rather than what the header implies.
intervals = raw.index.to_series().diff()
period = intervals.median() if SAMPLE_PERIOD == "auto" else pd.Timedelta(SAMPLE_PERIOD)
threshold = GAP_FACTOR * period

# A jump this much longer than the typical interval is an outage; the rest is jitter.
jumps = np.flatnonzero((intervals > threshold).to_numpy())
gaps = [(raw.index[j - 1] + period, raw.index[j]) for j in jumps]

if SAMPLE_PERIOD == "auto":
    # Keep every sample at the time it was taken, and mark each outage with one empty
    # row so the line breaks there.
    markers = pd.DataFrame(np.nan, columns=raw.columns,
                           index=pd.DatetimeIndex([start for start, _ in gaps],
                                                  tz=raw.index.tz))
    aq = pd.concat([raw, markers]).sort_index() if len(markers) else raw.copy()
else:
    aq = raw.reindex(pd.date_range(raw.index[0], raw.index[-1], freq=period))
    landed = int(aq.notna().any(axis=1).sum())
    if landed < len(raw):
        print(f"WARNING: {len(raw) - landed:,} of {len(raw):,} samples did not fall on "
              f'the {SAMPLE_PERIOD} grid and were dropped. Leave SAMPLE_PERIOD on "auto" '
              "to keep them.")
aq.index.name = "timestamp"

VALUE_COLS = [c for c in aq.columns if aq[c].notna().any()]
empty = [c for c in aq.columns if c not in VALUE_COLS]
if empty:
    print(f"columns with no readings at all: {', '.join(empty)}")

n_samples = int(aq[VALUE_COLS].notna().any(axis=1).sum())
span = aq.index[-1] - aq.index[0] + period
missing = sum((end - start for start, end in gaps), pd.Timedelta(0))
coverage = 1 - missing / span
cadence = f"{period.total_seconds():g} s"

print(f"\n{plural(len(frames), 'file')} merged")
print(f"span      {aq.index[0]} -> {aq.index[-1]}  ({span})")
print(f"cadence   one sample every {cadence}, "
      f"{'measured from the data' if SAMPLE_PERIOD == 'auto' else 'forced by SAMPLE_PERIOD'}")
print(f"samples   {n_samples:,}, covering {coverage:.1%} of the span")
aq.head()

In [ ]:
gap_table = pd.DataFrame(gaps, columns=["gap_start", "gap_end"])
gap_table["duration"] = (gap_table["gap_end"] - gap_table["gap_start"]
                         if len(gap_table) else pd.Series(dtype="timedelta64[ns]"))

# The stretches between the outages, for the coverage chart.
covered = list(zip([aq.index[0]] + [end for _, end in gaps],
                   [start for start, _ in gaps] + [aq.index[-1] + period]))

if len(gap_table):
    gap_table = gap_table.sort_values("duration", ascending=False).reset_index(drop=True)
    print(f"{plural(len(gap_table), 'gap')} longer than {threshold.total_seconds():g} s, "
          f"{missing} of missing time in total (gap_end is the first sample back on "
          "record). Longest first:")
    display(gap_table.head(10))
else:
    print(f"No gaps — no two consecutive samples are more than "
          f"{threshold.total_seconds():g} s apart.")

## 5. Plotting setup

Colours are assigned by what each series *is*, not by the order it happens to be
drawn in, so a variable keeps its colour across every figure below.

The four PM sizes are an **ordered** family — PM1.0 ⊂ PM2.5 ⊂ PM4.0 ⊂ PM10 — so they
get one hue stepped light→dark rather than four unrelated hues; size reads straight
off the shade. The two gas indices are separate identities and take the next
categorical hues. Both palettes pass the colour-vision-deficiency and lightness
checks in light and dark mode; because two of the steps sit below 3:1 against a light
surface, every line is also **directly labelled** at its right end, so identity never
rests on colour alone.

VOC and NOx are a 1–500 index and PM is µg/m³ — different quantities, so they get
their own charts rather than a second y-axis.

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

if DARK:
    SURFACE, TEXT, MUTED, GRID = "#1a1a19", "#ffffff", "#c3c2b7", "#33332f"
    # Stepped for the dark surface and run the other way, so the largest size is
    # still the strongest mark: contrast grows with the fraction in both modes.
    PM_RAMP = ["#184f95", "#2a78d6", "#5598e7", "#9ec5f4"]
    CATEGORICAL = ["#d95926", "#199e70", "#c98500", "#d55181", "#008300", "#9085e9"]
else:
    SURFACE, TEXT, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e2"
    PM_RAMP = ["#86b6ef", "#5598e7", "#2a78d6", "#184f95"]
    CATEGORICAL = ["#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]

PM_COLS = ["PM1.0_ug_m3", "PM2.5_ug_m3", "PM4.0_ug_m3", "PM10_ug_m3"]
LABELS = {
    "PM1.0_ug_m3": ("PM1.0", "µg/m³"), "PM2.5_ug_m3": ("PM2.5", "µg/m³"),
    "PM4.0_ug_m3": ("PM4.0", "µg/m³"), "PM10_ug_m3": ("PM10", "µg/m³"),
    "VOC_index": ("VOC index", "index"), "NOx_index": ("NOx index", "index"),
    "humidity_pct": ("Humidity", "%RH"), "temperature_C": ("Temperature", "°C"),
}

# Entity -> colour, fixed before anything is drawn so filtering a series out never
# repaints the others.
COLOR, _spare = {}, list(CATEGORICAL)
for col in VALUE_COLS:
    if col in PM_COLS:
        COLOR[col] = PM_RAMP[PM_COLS.index(col)]
    else:
        COLOR[col] = _spare.pop(0) if _spare else MUTED


def label_of(col):
    return LABELS.get(col, (col, ""))[0]


def unit_of(col):
    return LABELS.get(col, (col, ""))[1]


plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "figure.dpi": 120, "savefig.bbox": "tight",
    "font.size": 10, "text.color": TEXT,
    "axes.edgecolor": GRID, "axes.linewidth": 0.8,
    "axes.labelcolor": MUTED, "axes.labelsize": 10,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "xtick.direction": "out", "ytick.direction": "out",
    "grid.color": GRID, "grid.linewidth": 0.8, "grid.linestyle": "-",
    "legend.frameon": False, "legend.fontsize": 9, "legend.labelcolor": MUTED,
    "lines.linewidth": 1.4, "lines.solid_capstyle": "round",
    "axes.prop_cycle": plt.cycler(color=[PM_RAMP[2]] + CATEGORICAL),
})


def style(ax, title=None, subtitle=None, ylabel=None, grid="y"):
    """Hairline grid, two recessive spines, a left-aligned title and a muted subtitle."""
    ax.set_axisbelow(True)
    ax.grid(axis=grid, color=GRID, linewidth=0.8)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    if ylabel:
        ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title, loc="left", fontsize=12.5, fontweight="semibold",
                     color=TEXT, pad=22 if subtitle else 10)
    if subtitle:
        ax.annotate(subtitle, xy=(0, 1), xycoords="axes fraction", xytext=(0, 7),
                    textcoords="offset points", fontsize=9.5, color=MUTED, va="bottom")
    return ax


def label_end(ax, series, text, color, dx=6):
    """Direct-label a line at its last real value — the relief for low-contrast hues."""
    s = series.dropna()
    if s.empty:
        return
    ax.annotate(text, xy=(s.index[-1], s.iloc[-1]), xytext=(dx, 0),
                textcoords="offset points", color=color, fontsize=9,
                va="center", ha="left", clip_on=False, annotation_clip=False)


def time_axis(ax):
    """Date ticks that stay readable whether the run is two hours or two weeks."""
    locator = mdates.AutoDateLocator(minticks=4, maxticks=8)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    ax.set_xlabel("")


def guideline(ax, col):
    """WHO 24-hour guideline for this pollutant, as a labelled threshold.

    Skipped when the guideline sits far above everything recorded: a line at 45 over
    a run that never passed 14 says nothing and squashes the data into a strip.
    """
    if not SHOW_GUIDELINES or col not in WHO_24H:
        return
    y = WHO_24H[col]
    lo, hi = ax.get_ylim()               # data-driven, since this is called after plotting
    if y > hi * 1.25:
        return
    ax.axhline(y, color=MUTED, linewidth=0.8, linestyle=(0, (4, 3)), alpha=0.7, zorder=1)
    ax.set_ylim(lo, max(hi, y * 1.08))
    high = y > lo + 0.85 * (ax.get_ylim()[1] - lo)     # keep the label inside the axes
    ax.annotate(f"WHO 24-h {label_of(col)} {y:g}", xy=(0.004, y), xycoords=("axes fraction", "data"),
                xytext=(0, -3 if high else 3), textcoords="offset points", fontsize=8.5,
                color=MUTED, va="top" if high else "bottom", bbox=THRESHOLD_LABEL)


# Threshold labels sit on top of the data, so they carry the surface behind them.
THRESHOLD_LABEL = dict(facecolor=SURFACE, edgecolor="none", pad=1.5)


def finish(fig, name=None):
    fig.tight_layout()
    if SAVE_FIGS and name:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / f"{name}.png", dpi=200)
    plt.show()


plot_df = aq[VALUE_COLS].resample(RESAMPLE).mean() if RESAMPLE else aq[VALUE_COLS]
sampled = f"{RESAMPLE} means" if RESAMPLE else f"~{cadence} samples"
print(f"plotting {len(plot_df):,} points per series ({sampled})")

## 6. The run at a glance

In [ ]:
def stat_tiles(tiles, name=None):
    """A row of headline numbers. Some questions are a number, not a chart."""
    fig, axes = plt.subplots(1, len(tiles), figsize=(2.7 * len(tiles), 1.5))
    for ax, (value, caption, note) in zip(np.atleast_1d(axes), tiles):
        ax.axis("off")
        ax.text(0, 0.72, value, fontsize=23, fontweight="semibold", color=TEXT,
                va="center", ha="left")
        ax.text(0, 0.30, caption, fontsize=10, color=TEXT, va="center", ha="left")
        ax.text(0, 0.06, note, fontsize=9, color=MUTED, va="center", ha="left")
    finish(fig, name)


def fmt_delta(td):
    total = int(td.total_seconds())
    days, rem = divmod(total, 86400)
    hours, minutes = divmod(rem // 60, 60)
    return (f"{days}d {hours}h" if days else
            f"{hours}h {minutes}m" if hours else f"{minutes}m")


headline = "PM2.5_ug_m3" if "PM2.5_ug_m3" in VALUE_COLS else VALUE_COLS[0]
series = aq[headline]
peak_at = series.idxmax()

stat_tiles([
    (fmt_delta(span), "logged",
     f"{plural(len(inventory), 'file')}, {n_samples:,} samples"),
    (f"{coverage:.1%}", "coverage", f"{plural(len(gap_table), 'gap')} · one sample / {cadence}"),
    (f"{series.mean():.1f}", f"mean {label_of(headline)} {unit_of(headline)}",
     f"median {series.median():.1f} · p95 {series.quantile(0.95):.1f}"),
    (f"{series.max():.1f}", f"peak {label_of(headline)} {unit_of(headline)}",
     f"at {peak_at:%d %b %H:%M}" if pd.notna(peak_at) else "—"),
], name="00-headline")

In [ ]:
summary = pd.DataFrame({
    "samples": aq[VALUE_COLS].count(),
    "filled": aq[VALUE_COLS].count() / n_samples,
    "mean": aq[VALUE_COLS].mean(),
    "median": aq[VALUE_COLS].median(),
    "p95": aq[VALUE_COLS].quantile(0.95),
    "max": aq[VALUE_COLS].max(),
})
summary.insert(0, "unit", [unit_of(c) for c in summary.index])
summary.index = [label_of(c) for c in summary.index]
summary.index.name = "variable"
summary.round(2)

## 7. Coverage

Where the logger was actually recording. Anything not filled is time the merged
series carries as `NaN` — the reason every chart below breaks its line over a gap
instead of drawing through it.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 1.9))

spans = [(mdates.date2num(a), mdates.date2num(b) - mdates.date2num(a)) for a, b in covered]
ax.broken_barh(spans, (0.25, 0.5), facecolors=PM_RAMP[2], edgecolor="none")

# File boundaries: hairline ticks, so rotation seams are visible without labelling 45 files.
SURFACE_TICK_NOTE = "pale ticks are file boundaries" if DARK else "white ticks are file boundaries"
for t in inventory["start"]:
    ax.plot([mdates.date2num(t)] * 2, [0.25, 0.75], color=SURFACE, linewidth=1.0, zorder=3)

if len(gap_table):
    worst = gap_table.iloc[0]
    ax.annotate(f"longest gap {fmt_delta(worst.duration)}",
                xy=(mdates.date2num(worst.gap_start + worst.duration / 2), 0.78),
                xytext=(0, 4), textcoords="offset points", fontsize=9,
                color=MUTED, ha="center", va="bottom")

ax.set_ylim(0, 1.15)
ax.set_yticks([])
subtitle = f"{plural(len(inventory), 'file')} · {coverage:.1%} of the span has data"
if len(inventory) > 1:
    subtitle += f" · {SURFACE_TICK_NOTE}"
style(ax, "Recording coverage", subtitle, grid="x")
ax.spines["left"].set_visible(False)
time_axis(ax)
finish(fig, "01-coverage")

## 8. Particulate matter

The four sizes are cumulative — PM10 counts everything PM1.0 counts and more — so
the lines nest, and the spread between them is the coarse fraction.

In [ ]:
pm_cols = [c for c in PM_COLS if c in VALUE_COLS]

fig, ax = plt.subplots(figsize=(11, 4.4))
for col in pm_cols:
    ax.plot(plot_df.index, plot_df[col], color=COLOR[col], linewidth=1.4,
            label=label_of(col))
for col in ("PM2.5_ug_m3", "PM10_ug_m3"):
    if col in pm_cols:
        guideline(ax, col)

style(ax, "Particulate matter", f"{sampled} · µg/m³", ylabel="µg/m³")
time_axis(ax)
ax.legend(loc="upper left", ncols=len(pm_cols), bbox_to_anchor=(0, 1.02),
          handlelength=1.6, columnspacing=1.4)
x0, x1 = ax.get_xlim()
ax.set_xlim(x0, x1 + (x1 - x0) * 0.05)
for col in pm_cols:
    label_end(ax, plot_df[col], label_of(col), COLOR[col])
finish(fig, "02-pm")

## 9. PM2.5, zoomed

The overview above draws minute means across the whole run, which smooths away
anything shorter than a minute. This is one hour of PM2.5 at the raw 1 Hz the logger
writes, so individual samples are visible; the heavier line is a 1-minute rolling
mean through them, the same smoothing the overview applies.

In [ ]:
# The window to zoom into. ZOOM_DAY picks which day it lands on when the run spans
# several; None takes the first day whose window has data.
ZOOM_COL = "PM2.5_ug_m3"
ZOOM_DAY = None                       # e.g. "2026-09-02"
ZOOM_FROM, ZOOM_TO = "11:30", "12:30"


def as_day(value):
    """A midnight Timestamp on the merged index's own clock (tz-aware or not)."""
    ts = pd.Timestamp(value).normalize()
    return ts.tz_localize(aq.index.tz) if aq.index.tz and ts.tz is None else ts


candidates = ([as_day(ZOOM_DAY)] if ZOOM_DAY
              else list(pd.Index(aq.index.normalize().unique())))
zoom = None
for day in candidates:
    lo, hi = day + pd.Timedelta(f"{ZOOM_FROM}:00"), day + pd.Timedelta(f"{ZOOM_TO}:00")
    slice_ = aq.loc[lo:hi, ZOOM_COL]
    if slice_.notna().any():
        zoom = slice_
        break

if zoom is None:
    print(f"No {label_of(ZOOM_COL)} readings between {ZOOM_FROM} and {ZOOM_TO} on "
          f"{ZOOM_DAY or 'any day of this run'}.")
    print(f"The run covers {aq.index[0]:%Y-%m-%d %H:%M} to {aq.index[-1]:%Y-%m-%d %H:%M} "
          "— set ZOOM_DAY, ZOOM_FROM and ZOOM_TO to a window inside it.")
else:
    # A minute of samples, however many that is at this run's cadence.
    per_minute = max(3, round(60 / period.total_seconds()))
    smooth = zoom.rolling("60s", center=True, min_periods=per_minute // 2).mean()
    peak_at = zoom.idxmax()

    fig, ax = plt.subplots(figsize=(11, 3.8))
    ax.plot(zoom.index, zoom, color=COLOR[ZOOM_COL], linewidth=0.9, alpha=0.45,
            label=f"samples (~{cadence} apart)")
    ax.plot(smooth.index, smooth, color=COLOR[ZOOM_COL], linewidth=2.0,
            label="1 min rolling mean")
    guideline(ax, ZOOM_COL)

    # One direct label, on the thing a zoom is usually opened to look at.
    ax.plot([peak_at], [zoom.max()], marker="o", markersize=8, color=COLOR[ZOOM_COL],
            markeredgecolor=SURFACE, markeredgewidth=2, zorder=4)
    ax.annotate(f"peak {zoom.max():.1f} at {peak_at:%H:%M:%S}", xy=(peak_at, zoom.max()),
                xytext=(8, 2), textcoords="offset points", fontsize=9.5, color=TEXT,
                bbox=THRESHOLD_LABEL)

    style(ax, f"{label_of(ZOOM_COL)}, {ZOOM_FROM}–{ZOOM_TO}",
          f"{day:%a %d %b %Y} · {zoom.count():,} samples · "
          f"mean {zoom.mean():.1f} · range {zoom.min():.1f}–{zoom.max():.1f} "
          f"{unit_of(ZOOM_COL)}", ylabel=unit_of(ZOOM_COL))
    time_axis(ax)
    ax.margins(x=0.01)
    ax.legend(loc="upper left", ncols=2, bbox_to_anchor=(0, 1.02), handlelength=1.6,
              columnspacing=1.4)
    finish(fig, "07-pm25-zoom")

## 10. VOC and NOx indices

Both are a dimensionless 1–500 index where 100 is the sensor's rolling baseline for
its own environment: above 100 is worse than this room's recent normal, below is
better. NOx stays blank for the first minutes of a run while its index warms up, so
its line starts late.

In [ ]:
gas_cols = [c for c in ("VOC_index", "NOx_index") if c in VALUE_COLS]

if gas_cols:
    fig, ax = plt.subplots(figsize=(11, 3.8))
    for col in gas_cols:
        ax.plot(plot_df.index, plot_df[col], color=COLOR[col], linewidth=1.4,
                label=label_of(col))
    ax.axhline(100, color=MUTED, linewidth=0.8, linestyle=(0, (4, 3)), alpha=0.7, zorder=1)
    ax.annotate("baseline 100", xy=(0.004, 100), xycoords=("axes fraction", "data"),
                xytext=(0, 3), textcoords="offset points", fontsize=8.5, color=MUTED,
                va="bottom", bbox=THRESHOLD_LABEL)

    style(ax, "Gas indices", f"{sampled} · 1–500, 100 = the sensor's rolling baseline",
          ylabel="index")
    time_axis(ax)
    ax.legend(loc="upper left", ncols=len(gas_cols), bbox_to_anchor=(0, 1.02),
              handlelength=1.6, columnspacing=1.4)
    x0, x1 = ax.get_xlim()
    ax.set_xlim(x0, x1 + (x1 - x0) * 0.05)
    for col in gas_cols:
        label_end(ax, plot_df[col], label_of(col), COLOR[col])
    finish(fig, "03-gas")
else:
    print("No VOC/NOx columns in this run.")

## 11. Every variable, one panel each

Same time axis down the page, each variable on its own scale — the comparison a
second y-axis would fake.

In [ ]:
fig, axes = plt.subplots(len(VALUE_COLS), 1, sharex=True,
                         figsize=(11, 1.55 * len(VALUE_COLS)))
for ax, col in zip(np.atleast_1d(axes), VALUE_COLS):
    ax.plot(plot_df.index, plot_df[col], color=COLOR[col], linewidth=1.3)
    if col in PM_COLS:      # filling to zero only reads as area on a concentration
        ax.fill_between(plot_df.index, plot_df[col], color=COLOR[col], alpha=0.10,
                        linewidth=0)
    guideline(ax, col)
    style(ax, ylabel=unit_of(col))
    ax.annotate(label_of(col), xy=(0, 1), xycoords="axes fraction", xytext=(0, 4),
                textcoords="offset points", fontsize=10.5, fontweight="semibold",
                color=TEXT, va="bottom")
time_axis(np.atleast_1d(axes)[-1])
fig.align_ylabels(np.atleast_1d(axes))
finish(fig, "04-variables")

## 12. Hour-of-day profile

Folds the run onto a 24-hour clock: the median at each hour, with the middle half of
the readings shaded. It needs at least a day or two before it says anything about a
daily rhythm rather than about one afternoon.

In [ ]:
span = aq.index[-1] - aq.index[0]
if span < pd.Timedelta("6h"):
    print(f"Run is only {fmt_delta(span)} long — an hour-of-day profile needs at "
          "least a day to mean anything. Skipped.")
else:
    hours = aq.index.hour
    fig, ax = plt.subplots(figsize=(11, 3.6))
    col = headline
    grouped = aq[col].groupby(hours)
    med, lo, hi = grouped.median(), grouped.quantile(0.25), grouped.quantile(0.75)

    ax.fill_between(med.index, lo, hi, color=COLOR[col], alpha=0.16, linewidth=0)
    ax.plot(med.index, med, color=COLOR[col], linewidth=2.0)
    guideline(ax, col)

    peak_hour = med.idxmax()
    ax.plot([peak_hour], [med.max()], marker="o", markersize=8, color=COLOR[col],
            markeredgecolor=SURFACE, markeredgewidth=2, zorder=4)
    ax.annotate(f"{med.max():.1f} at {peak_hour:02d}:00", xy=(peak_hour, med.max()),
                xytext=(8, 2), textcoords="offset points", fontsize=9.5, color=TEXT)

    style(ax, f"{label_of(col)} by hour of day",
          f"median line, interquartile range shaded · {fmt_delta(span)} of data",
          ylabel=unit_of(col))
    ax.set_xlim(-0.5, 23.5)
    ax.set_xticks(range(0, 24, 3))
    ax.set_xticklabels([f"{h:02d}:00" for h in range(0, 24, 3)])
    finish(fig, "05-hour-of-day")

## 13. Distributions

How much of the run sat below any given concentration. Read it at the guideline
line: where the curve crosses it is the fraction of the time spent under it.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
for col in pm_cols:
    values = np.sort(aq[col].dropna().to_numpy())
    if not values.size:
        continue
    ax.plot(values, np.arange(1, values.size + 1) / values.size,
            color=COLOR[col], linewidth=1.6, label=label_of(col))

upper = max(aq[c].quantile(0.995) for c in pm_cols)
ax.set_xlim(0, upper)
ax.set_ylim(0, 1.02)
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0%", "25%", "50%", "75%", "100%"])
for col in ("PM2.5_ug_m3", "PM10_ug_m3"):
    if col in pm_cols:
        y = WHO_24H.get(col)
        if SHOW_GUIDELINES and y is not None and y < upper:
            ax.axvline(y, color=MUTED, linewidth=0.8, linestyle=(0, (4, 3)), alpha=0.7)
            share = (aq[col].dropna() < y).mean()
            ax.annotate(f"WHO {label_of(col)} {y:g} — {share:.0%} below", xy=(y, 0.02),
                        xytext=(4, 0), textcoords="offset points", fontsize=8.5,
                        color=MUTED, bbox=THRESHOLD_LABEL)

style(ax, "Share of the run below a given concentration",
      f"empirical CDF · the top 0.5% of samples run past the right edge "
      f"({upper:.0f} µg/m³)", ylabel="share of samples")
ax.set_xlabel("µg/m³")
ax.legend(loc="center right", handlelength=1.6)

# One direct label, on the series that carries the story; the legend names the rest.
if headline in pm_cols:
    s = aq[headline].dropna()
    if s.size:
        ax.annotate(f"median {label_of(headline)} {s.median():.1f}",
                    xy=(s.median(), 0.5), xytext=(6, -14), textcoords="offset points",
                    fontsize=9, color=COLOR[headline], bbox=THRESHOLD_LABEL)
finish(fig, "06-distribution")

## 14. Export

One CSV holding the whole run on its continuous index: `timestamp` as an absolute
local wall-clock time, one row per sample, and an empty row at each outage.

In [ ]:
aq.to_csv(MERGED_CSV, float_format="%.1f")
print(f"wrote {MERGED_CSV.resolve()}  ({MERGED_CSV.stat().st_size / 1e6:.1f} MB, "
      f"{len(aq):,} rows)")

# A resampled copy is a fraction of the size and is usually what you want to share.
hourly = aq[VALUE_COLS].resample("1h").mean().round(2)
hourly.to_csv(MERGED_CSV.with_name(MERGED_CSV.stem + "_hourly.csv"))
hourly.head()

---

### Notes

- **Times are local wall-clock with no timezone.** The device does not know its
  offset; set `TIMEZONE` to attach one, which labels the times without shifting
  them.
- **Empty cells mean the sensor gave no reading**, which is not the same as zero.
  Means and medians here skip them; the coverage column above says how often it
  happened.
- **A stepped clock shows up as a duplicate timestamp**, and the merge reports
  those rather than smoothing over them — the periodic RTC resync can move the clock
  by a second or two, which is the correction working as intended.
- **The cadence is measured, not assumed.** The header describes a run of one row
  per sample; if that turns out to be every two seconds rather than every one, the
  merge follows the data and says so in its summary.